# 06 — Cross-approach evaluation

Loads `data/results/approach_{a,b,c}_all.parquet` and produces the thesis comparison table + per-category breakdown + latency / accuracy scatter plot.

**All approaches evaluated on the same test split** (`data/splits/test.parquet`) so cross-approach F1 comparison is valid.


In [ ]:
import sys
from pathlib import Path
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(AI_ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from shared.metrics import per_class_report, confusion_df, summarize_run

RESULTS = AI_ROOT / 'data' / 'results'

In [ ]:
frames = []
for name in ('approach_a_all', 'approach_b_all', 'approach_c_all'):
    p = RESULTS / f'{name}.parquet'
    if p.exists():
        frames.append(pd.read_parquet(p))
        print('loaded', p, len(frames[-1]))
    else:
        print('missing:', p)
all_df = pd.concat(frames, ignore_index=True)
print('total rows:', len(all_df))

In [ ]:
# Master comparison table — one row per (approach, model).
rows = []
for model, sub in all_df.groupby('model'):
    rows.append(summarize_run(model, sub['gold'].tolist(), sub['pred'].tolist(), sub['latency_ms'].tolist()))
summary = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
summary

In [ ]:
# Latency × accuracy Pareto plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(summary['p50_ms'], summary['macro_f1'], s=80)
for _, r in summary.iterrows():
    ax.annotate(r['approach'], (r['p50_ms'], r['macro_f1']),
                xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('p50 latency (ms, log scale)')
ax.set_ylabel('Macro-F1')
ax.set_title('Quality vs latency — all approaches')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'pareto_quality_vs_latency.png', dpi=120)
plt.show()

In [ ]:
# Per-category drilldown for the best model in each approach family.
for fam, prefix in [('A (zero-shot)', 'A:'), ('B (embedding)', 'B:'), ('C (few-shot)', 'C:')]:
    sub = summary[summary['approach'].str.startswith(prefix)]
    if sub.empty: continue
    best = sub.iloc[0]['approach']
    rows = all_df[all_df['model'] == best]
    print(f'\n=== {fam} — best: {best} ===')
    display(per_class_report(rows['gold'].tolist(), rows['pred'].tolist()))

In [ ]:
# Confusion matrices side by side for the top model per family
from shared.types import ALL_CATEGORIES
import seaborn as sns

fams = []
for fam, prefix in [('A', 'A:'), ('B', 'B:'), ('C', 'C:')]:
    sub = summary[summary['approach'].str.startswith(prefix)]
    if sub.empty: continue
    fams.append((fam, sub.iloc[0]['approach']))

fig, axes = plt.subplots(1, len(fams), figsize=(5*len(fams), 4))
if len(fams) == 1: axes = [axes]
for ax, (fam, model) in zip(axes, fams, strict=True):
    rows = all_df[all_df['model'] == model]
    cm = confusion_df(rows['gold'].tolist(), rows['pred'].tolist())
    sns.heatmap(cm.values, annot=True, fmt='d', cbar=False, ax=ax,
                xticklabels=ALL_CATEGORIES, yticklabels=ALL_CATEGORIES, cmap='Blues')
    ax.set_title(f'{fam}: {model}')
    ax.set_xlabel('predicted'); ax.set_ylabel('gold')
plt.tight_layout()
plt.savefig(RESULTS / 'confusion_per_family.png', dpi=120)
plt.show()

In [ ]:
# Persist the master table for the thesis chapter
summary.to_csv(RESULTS / 'summary_all_approaches.csv', index=False)
print('wrote', RESULTS / 'summary_all_approaches.csv')
summary